# LLM Enrichment Pipeline Walkthrough

This notebook walks through a simple AI-powered data engineering pattern: taking raw review text and turning it into structured columns you can analyze. By the end, you will know how to load the CSV, enrich one review, enrich a batch, and save the result as Parquet.

## Setup

1. Install dependencies with `pip install -r requirements.txt`.
2. Copy `.env.example` to `.env`.
3. Add your `OPENAI_API_KEY` to the `.env` file.

If you are in Jupyter already, you can also run the command below in a cell.

In [ ]:
from pathlib import Path
import subprocess
import sys

requirements_path = Path("requirements.txt")
if not requirements_path.exists():
    requirements_path = Path("../requirements.txt")

subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements_path)], check=True)


## Load data

The raw file has four columns:
- `review_id`: unique identifier
- `product_name`: product reviewed
- `review_text`: messy free-form customer text
- `date`: review date


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.ingest import load_reviews

data_path = PROJECT_ROOT / 'data' / 'reviews.csv'
reviews_df = load_reviews(str(data_path))
reviews_df.head()


## Single enrichment

Start with one row so you can inspect the structured output before paying to process the full dataset.

In [ ]:
from src.llm import get_client
from src.enrich import enrich_single

client = get_client()
single_result = enrich_single(reviews_df.loc[0, 'review_text'], client=client)
single_result


## Batch enrichment

Now enrich the full dataset or start with a smaller sample if you want a cheaper test run.

In [ ]:
from src.enrich import enrich_batch

enriched_df = enrich_batch(reviews_df, client=client, sample=10)
enriched_df.head()


## Analysis

Because the LLM output is now structured, simple pandas groupby operations become useful immediately.

In [ ]:
enriched_df.groupby('sentiment').size().sort_values(ascending=False)


In [ ]:
enriched_df.groupby('category').size().sort_values(ascending=False)


## Save output

Parquet is a better fit than CSV for analytics pipelines because it preserves types, compresses well, and is optimized for columnar reads.

In [ ]:
output_path = PROJECT_ROOT / 'output' / 'enriched_from_notebook.parquet'
enriched_df.to_parquet(output_path, index=False)
print(f'Saved enriched data to {output_path}')


## What's next

To adapt this pattern to Databricks, replace the pandas batch loop with PySpark `mapInPandas`, keep the extraction schema the same, and make sure each partition initializes an LLM client once instead of once per row.